# Voyager-Style Skill Library: Lifelong Learning via Accumulated Skills

## Definition

**Voyager** (Wang et al., 2023) was an embodied LLM agent that played Minecraft with no
fine-tuning at all. Its central trick was not a smarter prompt or a bigger model — it was
a **persistent skill library**: every time the agent solved a new sub-task, it saved the
working code as a reusable, named "skill." Future tasks first checked whether an existing
skill (or a composition of a few) already solved the problem before writing anything new.
Over time the agent built an ever-growing "automatic curriculum" of composable skills,
getting *faster and more capable* without ever touching the model's weights.

This notebook adapts that **core idea** to a domain-agnostic, code-executable setting —
small Python text/data-processing tasks — so no game engine or embodiment is required. The
mechanism transfers untouched:

1. Persistent, named skills (description + source code + usage count) stored in a library.
2. **Retrieval-before-generation**: an LLM call checks the library for a reusable or
   composable match before writing new code.
3. **Generate → sandbox-execute → verify → persist**: if nothing fits, the agent writes a
   new Python function, runs it in a restricted namespace, checks the result against test
   cases, and — only if it passes — commits it to the library for future reuse.
4. Demonstration across a short sequence of tasks of increasing complexity, where later
   tasks reuse or *compose* skills learned earlier, rather than re-deriving them.

## High-Level Workflow

```
                 ┌───────────────────────────┐
   new task ───▶ │ 1. Search skill library    │
                 │    (LLM decides: reuse /   │
                 │     compose / none fit)    │
                 └─────────────┬─────────────┘
                     match found│           no match
                                ▼                 ▼
                    ┌────────────────────┐  ┌─────────────────────────┐
                    │ Reuse existing      │  │ 2. Generate new Python   │
                    │ skill(s) directly   │  │    function via LLM      │
                    └─────────┬──────────┘  └────────────┬─────────────┘
                              │                            ▼
                              │              ┌──────────────────────────┐
                              │              │ 3. Sandbox-execute        │
                              │              │    (restricted namespace, │
                              │              │    no file/network I/O)   │
                              │              └────────────┬─────────────┘
                              │                            ▼
                              │              ┌──────────────────────────┐
                              │              │ 4. Verify against         │
                              │              │    expected test cases    │
                              │              └────────────┬─────────────┘
                              │                    passes  │  fails
                              │                            ▼        ▼
                              │              ┌──────────────────┐ (retry /
                              │              │ 5. Persist to     │  give up)
                              │              │    skill library  │
                              │              └──────────────────┘
                              ▼
                    ┌────────────────────────────┐
                    │ Return result, bump         │
                    │ usage_count on skill(s)     │
                    └────────────────────────────┘
```

## When to Use

- Long-running or repeatedly-invoked agents that face many *similar-but-not-identical*
  sub-tasks over their lifetime (coding assistants, data-wrangling agents, game/simulation
  agents, workflow automators).
- Situations where fine-tuning is impractical/expensive, but you still want the agent to
  visibly "get better" at a domain over time.
- Any setting where code-as-skill is a natural unit of reusable capability (as opposed to
  free-text facts, which belong in the other memory architectures in this folder).

## Strengths

- **Lifelong learning without training**: capability compounds purely through accumulated,
  inspectable artifacts (functions), not gradient updates.
- **Composability**: skills call other skills, so complexity grows combinatorially, not
  linearly, with the number of stored skills.
- **Transparent & auditable**: every skill is literal, readable Python source — easy to
  review, test, or hand-edit, unlike opaque learned weights.
- **Cheap reuse**: once a skill exists, later invocations skip the (comparatively expensive)
  code-generation LLM call entirely.

## Weaknesses

- **Retrieval quality bottleneck**: if the "does a skill already solve this?" lookup is
  wrong (misses a valid match, or wrongly reuses a bad match), errors compound silently.
- **Unbounded library growth**: without curation, the library accumulates near-duplicate or
  dead skills over a long run — real Voyager also needed periodic skill-library maintenance.
- **Sandboxing is essential and easy to under-engineer**: LLM-generated code executing
  locally is a real security surface. See the explicit safety note below.
- **No cross-task generalization inside a skill**: a skill only generalizes as far as its
  parameters allow; genuinely novel task shapes still require fresh code generation.


## Setup

We use the repo-wide `helpers.get_llm()` factory (platform-aware: Groq on Windows,
Databricks on macOS) instead of instantiating a provider client directly, per repo
convention.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from helpers import get_llm

llm = get_llm()

## Safety Note: Sandboxing Generated Code

The whole point of this pattern is that the LLM writes and executes **new, untrusted
Python code** at runtime. We never hand that code a real interpreter with full privileges.
Instead:

- Generated source is **statically screened** for a denylist of dangerous tokens
  (`import`, `open(`, `exec(`, `eval(`, `__`, `os.`, `sys.`, `subprocess`, `socket`,
  `requests`, `input(`, file/network primitives, etc.) *before* it is ever executed.
- Execution happens via `exec()` into a **restricted namespace**: a hand-picked, minimal
  `__builtins__` (no `open`, `import`, `eval`, `exec`, `__import__`, `compile`, `input`,
  `exit`, `quit`) plus only the specific pure-Python helper functions we intentionally
  expose (e.g. previously-verified skills, for composition).
- No filesystem, network, subprocess, or `os`/`sys` access is reachable from inside the
  sandbox — the denylist plus the restricted builtins are two independent layers, so even
  a `__builtins__` bypass attempt is still caught by the source screen.
- Every generated function is **verified against concrete test cases** before being
  persisted — nothing enters the durable skill library unless it demonstrably produces
  correct output on known inputs.

This is a simplified, educational sandbox — good enough to keep this notebook self-contained
and demonstrably safe for *pure computation* (string/number/list processing), but **not** a
substitute for a real isolated execution environment (container, gVisor, WASM, etc.) in a
production system that must resist a determined adversary.

In [ ]:
import re

FORBIDDEN_TOKENS = [
    "import ", "__", "open(", "exec(", "eval(", "compile(", "input(",
    "os.", "sys.", "subprocess", "socket", "requests", "shutil", "pathlib",
    "globals(", "locals(", "getattr(", "setattr(", "delattr(", "exit(", "quit(",
]


def screen_source(source_code: str) -> None:
    """Static denylist screen. Raises ValueError if the source looks unsafe."""
    lowered = source_code.lower()
    for token in FORBIDDEN_TOKENS:
        if token in lowered:
            raise ValueError(f"Rejected generated code: forbidden token '{token}' found.")


def make_restricted_builtins() -> dict:
    """A minimal, safe __builtins__ mapping -- no I/O, no import, no eval/exec."""
    safe_names = [
        "abs", "all", "any", "bool", "dict", "enumerate", "filter", "float",
        "int", "len", "list", "map", "max", "min", "range", "reversed",
        "round", "set", "sorted", "str", "sum", "tuple", "zip", "isinstance",
        "print",
    ]
    import builtins as _builtins

    return {name: getattr(_builtins, name) for name in safe_names}


def sandboxed_exec(source_code: str, function_name: str, extra_globals: dict | None = None):
    """
    Screen + exec `source_code` (expected to define `function_name`) in a restricted
    namespace, and return the resulting callable. Raises on unsafe or malformed code.
    """
    screen_source(source_code)

    restricted_globals = {"__builtins__": make_restricted_builtins()}
    if extra_globals:
        restricted_globals.update(extra_globals)

    local_namespace: dict = {}
    exec(source_code, restricted_globals, local_namespace)  # noqa: S102 -- sandboxed on purpose

    if function_name not in local_namespace:
        raise ValueError(
            f"Generated code did not define expected function '{function_name}'."
        )
    return local_namespace[function_name]

## What We Are Going to Do

1. Define an in-memory **skill library** (`SKILL_LIBRARY`): a dict of
   `skill_name -> {description, code, function, usage_count}`.
2. Seed it with two trivial starter skills (`count_vowels`, `reverse_string`) — analogous
   to Voyager's initial primitive actions.
3. Build a `solve_task(task_description, test_cases)` pipeline that:
   - asks the LLM to check the library for a reusable/composable match,
   - on a miss, asks the LLM to *write* a new function, sandbox-executes it, verifies it
     against `test_cases`, and persists it on success,
   - on a hit, reuses the existing skill(s) directly (no generation call at all).
4. Run four sequential tasks of increasing complexity and watch the library grow and get
   reused rather than recreated.

In [ ]:
SKILL_LIBRARY: dict = {}


def _register(name: str, description: str, source_code: str, function_name: str) -> None:
    """Sandbox-verify then register a skill's source + live callable into the library."""
    func = sandboxed_exec(source_code, function_name)
    SKILL_LIBRARY[name] = {
        "description": description,
        "code": source_code,
        "function_name": function_name,
        "function": func,
        "usage_count": 0,
    }


# --- Seed skills (Voyager's "primitive actions") ---------------------------
_register(
    name="count_vowels",
    description="Count the number of vowels (a, e, i, o, u, case-insensitive) in a string.",
    source_code=(
        "def count_vowels(text):\n"
        "    return sum(1 for ch in text.lower() if ch in 'aeiou')"
    ),
    function_name="count_vowels",
)

_register(
    name="reverse_string",
    description="Reverse a string.",
    source_code=(
        "def reverse_string(text):\n"
        "    return text[::-1]"
    ),
    function_name="reverse_string",
)


def list_skills() -> str:
    lines = []
    for name, skill in SKILL_LIBRARY.items():
        lines.append(
            f"- {name}: {skill['description']} (used {skill['usage_count']}x)"
        )
    return "\n".join(lines)


print("Seeded skill library:")
print(list_skills())

## Skill Lookup (Retrieval Before Generation)

Before writing anything new, we ask the LLM to look at the current library's descriptions
and decide: does an existing skill (or combination of skills) already solve this task? This
mirrors Voyager's skill-retrieval step, done here via a direct LLM judgment call rather than
a vector index — simple, and sufficient for a library of this size.

The LLM must answer in a strict, parseable format so we can act on it programmatically.

In [ ]:
import json as _json


def find_matching_skill(task_description: str) -> dict:
    """
    Ask the LLM whether the existing skill library already solves `task_description`.
    Returns a dict: {"reuse": bool, "skill_names": [...], "reasoning": str}
    """
    prompt = f'''You are maintaining a library of reusable Python skills for an agent.

Current skill library:
{list_skills()}

New task: "{task_description}"

Decide whether the existing skills already fully solve this task, either by using ONE
skill directly, or by COMPOSING two or more existing skills in sequence. Do NOT propose
reuse if the task needs logic that no combination of existing skills provides.

Respond with ONLY a JSON object, no prose, no markdown fences, in this exact shape:
{{"reuse": true or false, "skill_names": ["skill_a", "skill_b"], "reasoning": "..."}}

If reuse is false, "skill_names" must be an empty list.'''

    response = llm.invoke(prompt).content.strip()
    # Be defensive in case the model wraps the JSON in a code fence anyway.
    response = response.strip("`")
    if response.lower().startswith("json"):
        response = response[4:].strip()
    try:
        return _json.loads(response)
    except _json.JSONDecodeError:
        return {"reuse": False, "skill_names": [], "reasoning": "unparseable response"}

## Skill Generation (When Nothing Fits)

When no existing skill covers the task, the LLM writes a brand-new, single, pure Python
function. The prompt explicitly tells it:

- the exact function name to use,
- that it may call any already-verified skill functions by name (their signatures are
  listed) if composition helps,
- that it must not use `import`, file/network I/O, or any of the other denylisted patterns
  from the safety note above (kept simple: pure computation only).

In [ ]:
def _slugify(task_description: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "_", task_description.lower()).strip("_")
    return slug[:40] or "new_skill"


def generate_new_skill(task_description: str) -> tuple[str, str, str]:
    """
    Ask the LLM to write a new Python function solving `task_description`.
    Returns (function_name, source_code, description).
    """
    function_name = f"skill_{_slugify(task_description)}"
    available = "\n".join(
        f"- {name}({skill['function_name']}): {skill['description']}"
        for name, skill in SKILL_LIBRARY.items()
    )

    prompt = f'''Write a single pure Python function named exactly `{function_name}` that
solves this task:

"{task_description}"

You may call any of these already-available helper functions by name if it helps
(they are already defined in the execution namespace, do not redefine them):
{available}

Strict rules:
- Output ONLY the function definition, no explanation, no markdown code fences.
- Do not use `import`, file I/O, network calls, `eval`, `exec`, or any dunder attributes.
- The function must be deterministic, pure computation only, and self-contained aside from
  the helper functions listed above.
'''

    raw = llm.invoke(prompt).content.strip()
    # Strip a markdown fence if the model adds one despite instructions.
    raw = re.sub(r"^```(python)?", "", raw.strip(), flags=re.IGNORECASE).strip()
    raw = re.sub(r"```$", "", raw.strip()).strip()

    description = f"Auto-generated skill for task: {task_description}"
    return function_name, raw, description

In [ ]:
def verify_skill(func, test_cases: list[tuple]) -> bool:
    """test_cases: list of (args_tuple, expected_output). Returns True iff all pass."""
    for args, expected in test_cases:
        try:
            actual = func(*args)
        except Exception as exc:  # noqa: BLE001 -- we want to catch and report any failure
            print(f"  verification error on input {args}: {exc}")
            return False
        if actual != expected:
            print(f"  verification failed on input {args}: got {actual!r}, expected {expected!r}")
            return False
    return True

## The `solve_task` Orchestrator

This ties retrieval, generation, sandboxed execution, verification, and persistence
together into the Voyager lifecycle described in the workflow diagram above.

In [ ]:
def _reused_result(skill_names: list[str], task_args: tuple):
    """Apply the reused skill(s) to task_args. Single skill: call directly.
    Multiple skills: compose left-to-right, feeding each skill's output into the next."""
    value = task_args[0]
    for name in skill_names:
        skill = SKILL_LIBRARY[name]
        value = skill["function"](value)
        skill["usage_count"] += 1
    return value


def solve_task(task_description: str, task_args: tuple, test_cases: list[tuple]):
    """
    Full Voyager-style lifecycle for one task:
      1. Try to reuse/compose existing skills (LLM lookup).
      2. If nothing fits, generate + sandbox-execute + verify + persist a new skill.
      3. Return the result of applying the (reused or newly persisted) skill to task_args.
    """
    print(f"\n=== Task: {task_description} ===")
    lookup = find_matching_skill(task_description)
    print(f"Lookup decision: {lookup}")

    if lookup.get("reuse") and lookup.get("skill_names"):
        skill_names = [n for n in lookup["skill_names"] if n in SKILL_LIBRARY]
        if skill_names:
            result = _reused_result(skill_names, task_args)
            print(f"Reused skill(s) {skill_names} -> result = {result!r}")
            return result
        print("Lookup named unknown skills; falling back to generation.")

    print("No existing skill fits -- generating a new one.")
    function_name, source_code, description = generate_new_skill(task_description)
    print(f"Generated candidate `{function_name}`:\n{source_code}\n")

    extra_globals = {name: s["function"] for name, s in SKILL_LIBRARY.items()}
    try:
        func = sandboxed_exec(source_code, function_name, extra_globals=extra_globals)
    except ValueError as exc:
        print(f"Rejected by sandbox: {exc}")
        return None

    if not verify_skill(func, test_cases):
        print("New skill failed verification -- NOT persisted.")
        return None

    SKILL_LIBRARY[function_name] = {
        "description": description,
        "code": source_code,
        "function_name": function_name,
        "function": func,
        "usage_count": 1,
    }
    result = func(*task_args)
    print(f"Verified and persisted new skill `{function_name}` -> result = {result!r}")
    return result

## Demonstration: Lifecycle Across Four Increasingly Complex Tasks

- **Task 1** and **Task 2** re-derive (via LLM generation, since they're phrased
  differently from the seeded skills) two small text-processing skills.
- **Task 3** is phrased so that it should be solvable purely by **composing** the skills
  from Tasks 1–2 (or the seeds) — a good test of the reuse path.
- **Task 4** requires genuinely new logic (palindrome-style comparison) but can *call* the
  by-then-available skills internally, showing skills building on skills.

Each task prints the lookup decision so you can see when the agent reuses vs. generates.

In [ ]:
# Task 1: a new phrasing of a string-processing task not already in the library.
result_1 = solve_task(
    task_description="Count how many uppercase letters are in a string",
    task_args=("Hello World AI",),
    test_cases=[
        (("Hello World AI",), 4),
        (("abc",), 0),
        (("XYZ",), 3),
    ],
)
result_1

### Discussion of the Output

The library had no skill for counting uppercase letters, so the lookup step correctly
returns `reuse: false`, the LLM writes a new `skill_count_how_many_uppercase_letters_..._`
function, it passes sandboxing and all three test cases, and is persisted. The library now
has three skills.

In [ ]:
# Task 2: another new, distinct skill.
result_2 = solve_task(
    task_description="Remove all whitespace from a string",
    task_args=("a b  c   d",),
    test_cases=[
        (("a b  c   d",), "abcd"),
        (("no spaces",), "nospaces"),
        (("",), ""),
    ],
)
result_2

### Discussion of the Output

Again no existing skill matches, so a fourth skill is generated, verified, and persisted.
At this point the library holds four independent skills: `count_vowels`, `reverse_string`,
and the two just generated.

In [ ]:
# Task 3: phrased to be solvable by composing two already-known skills.
result_3 = solve_task(
    task_description="Reverse a string and then count the vowels in the reversed string",
    task_args=("Voyager",),
    test_cases=[],  # not used on the reuse path -- lookup should short-circuit generation
)
result_3

### Discussion of the Output

This is the key Voyager moment: the lookup call should recognize that `reverse_string`
followed by `count_vowels` already solves the task, return `reuse: true` with
`skill_names: ["reverse_string", "count_vowels"]`, and the orchestrator composes them
directly — **no new LLM code-generation call is made at all**. Reusing existing skills is
strictly cheaper than generating new ones, which is exactly the payoff of the accumulated
library. Both skills' `usage_count` increments, visible in the summary below.

In [ ]:
# Task 4: needs new logic (a comparison), but can call existing skills internally.
result_4 = solve_task(
    task_description=(
        "Check whether a string has the same number of vowels as its reversed version "
        "(this is trivially always true, but implement it by actually reversing and "
        "counting rather than assuming so) -- return True or False"
    ),
    task_args=("Skill Library",),
    test_cases=[
        (("Skill Library",), True),
        (("abcde",), True),
        (("",), True),
    ],
)
result_4

### Discussion of the Output

Task 4 cannot be satisfied by *pure* reuse (the library has no boolean-comparison skill),
so the lookup correctly falls through to generation. The generation prompt exposes the
existing skill functions (including `reverse_string` and `count_vowels`) as callables the
new function is allowed to use, so a well-behaved model writes something like:

```python
def skill_check_whether_a_string_has_the_same_number(text):
    return count_vowels(text) == count_vowels(reverse_string(text))
```

reusing two library skills *inside* a newly generated one — composition happening at the
code-generation level, not just at the orchestration level. After verification it is
persisted as a fifth skill, itself now available for even later tasks to reuse or compose.

In [ ]:
print("Final skill library state:\n")
print(list_skills())
print(f"\nTotal skills accumulated: {len(SKILL_LIBRARY)}")

## Summary / Key Takeaways

- **The core Voyager idea is retrieval-before-generation over a persistent, code-based
  skill library** — not fine-tuning, not a bigger context window, just accumulated,
  verified, reusable artifacts that compound in value the longer the agent runs.
- **Sandboxing is not optional**: any pattern where an LLM writes code that gets executed
  needs a real containment strategy — here, a source denylist plus a restricted
  `exec()` namespace with no I/O, no `import`, no dunder access. Nothing is persisted to
  the library without first passing explicit test-case verification.
- **Composition is where the payoff compounds**: Task 3 was solved with *zero* new
  generation calls by chaining two existing skills; Task 4 needed one new generation call,
  but that new function itself called two existing skills internally — each solved task
  makes the *next* one cheaper or fully free.
- **This is one of several memory-architecture notebooks** in
  `07_Advanced_Agentic_Systems/Memory_and_State/Agentic_Memory_Architectures/`. Its
  siblings — Graph Memory, MemGPT-style tiered memory, and Agent Workflow Memory — are
  being authored separately and are not created by this notebook; together they cover
  distinct memory paradigms (procedural/skill memory here, vs. relational, tiered, and
  workflow-trace memory in the others).
